In [5]:
#importar dataset 
import pandas as pd

df_años = pd.read_csv("NLP\datasets_scraping/dataset_final_con_años.csv")
df_años.head()

,artist,song,year
0,moonspell,Heartshaped Abyss,NaN
1,moonspell,Let The Children Cum To Me...,2008
2,moonspell,Memento Mori,2020
3,moonspell,Once It Was Ours!,2020
4,moonspell,Serpent Angel,2024


In [10]:
df_letras = pd.read_csv("NLP/dataset_fusionado.csv")
df_letras.head(10)

,artist,song,lyrics,genres,genre_1,genre_2,genre_3,<<<<<<< HEAD
0,/moonspell/,Heartshaped Abyss,Never resist\nThe firewalker\nHe has been sent...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
1,/moonspell/,Let The Children Cum To Me...,"Hey you little Jesus' bride, why have you smil...","gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
2,/moonspell/,Memento Mori,"I am the moment, the soul\nThe moment that end...","gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
3,/moonspell/,Once It Was Ours!,Vanishing act inside the weak\nIn need of you ...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
4,/moonspell/,Serpent Angel,Father Satan send the Serpent\nPoison me with ...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
5,/moonspell/,The Darkening,"And we travel into the darkening,\na couldn't ...","gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
6,/moonspell/,Upon The Blood Of Men,Hate is the place where a man finally is\nThat...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
7,/running-wild/,White Buffalo,Acid rain and poison cauterize its skin\nA boi...,"heavy metal, power metal, speed metal",heavy metal,power metal,speed metal,NaN
8,/running-wild/,Billy The Kid,Silence\nIn the steps of no-man's land\nCamp f...,"heavy metal, power metal, speed metal",heavy metal,power metal,speed metal,NaN
9,/running-wild/,Branded And Exiled,Red hot iron heated by living coal\nReady for ...,"heavy metal, power metal, speed metal",heavy metal,power metal,speed metal,NaN


In [11]:
import pandas as pd
import numpy as np
import re
import ast
import html
from pathlib import Path
from tqdm.auto import tqdm
from transformers import pipeline

# ============================================================
# 1. CONFIGURACIÓN Y CARGA
# ============================================================
DATA_PATH = "NLP/dataset_fusionado.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

MACRO_GENRES = [
    "Rock", "Pop", "Country", "Electronic / Dance", "Folk / Traditional",
    "R&B / Soul / Funk", "Punk / Emo / Hardcore", "Metal", "Hip Hop / Rap",
    "Experimental", "Jazz", "Blues", "Latin", "Reggae / Caribbean",
    "Classical / Art Music", "Other"
]

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Shape inicial: {df.shape}")
except FileNotFoundError:
    raise FileNotFoundError(f"No se encontró el archivo en {DATA_PATH}")

# Limpieza inicial de columnas
df = df.drop(columns=["<<<<<<< HEAD"], errors="ignore")
df.columns = df.columns.str.strip().str.lower()

# ============================================================
# 2. FUNCIONES DE LIMPIEZA
# ============================================================
def clean_artist(text):
    if pd.isna(text): return np.nan
    text = html.unescape(str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    text = text.replace("/", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text if text != "" else np.nan

def normalize_raw_genre(genre):
    if pd.isna(genre): return None
    genre = html.unescape(str(genre)).strip().lower().strip("'").strip('"').replace("_", " ")
    genre = re.sub(r"\s+", " ", genre).strip().strip(" .;:,")
    if genre in ["", "nan", "none", "null", "unknown", "[]", "-", "n/a", "na"]: return None
    return genre

def parse_raw_genre_cell(value):
    if pd.isna(value): return []
    value = str(value).strip()
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            parsed_values = list(parsed) if isinstance(parsed, (list, tuple, set)) else [value]
        except: parsed_values = [value]
    elif "," in value:
        parsed_values = value.split(",")
    else:
        parsed_values = [value]
    
    normalized = []
    seen = set()
    for g in parsed_values:
        gn = normalize_raw_genre(g)
        if gn and gn not in seen:
            normalized.append(gn)
            seen.add(gn)
    return normalized

# ============================================================
# 3. CLASIFICACIÓN CON TRANSFORMER (Zero-Shot)
# ============================================================
print("Cargando modelo Transformer...")
genre_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Extraer géneros únicos para no clasificar 10,000 veces lo mismo
all_raw_genres = set()
for col in ["genres", "genre_1", "genre_2", "genre_3"]:
    if col in df.columns:
        for val in df[col].dropna():
            all_raw_genres.update(parse_raw_genre_cell(val))

all_raw_genres = sorted(list(all_raw_genres))
print(f"Detectados {len(all_raw_genres)} géneros únicos. Iniciando clasificación...")

genre_to_macro = {}
for genre in tqdm(all_raw_genres, desc="Clasificando géneros"):
    res = genre_classifier(
        sequences=f"The music genre is {genre}.",
        candidate_labels=MACRO_GENRES,
        hypothesis_template="This music genre belongs to {}.",
        multi_label=False
    )
    # Si la confianza es baja (< 0.30), asignamos "Other"
    label = res["labels"][0] if res["scores"][0] >= 0.30 else "Other"
    genre_to_macro[genre] = label

# ============================================================
# 4. APLICACIÓN DE RESULTADOS Y FILTRADO
# ============================================================
def get_macro_genres(value):
    raws = parse_raw_genre_cell(value)
    macros = [genre_to_macro.get(r) for r in raws if genre_to_macro.get(r)]
    return list(dict.fromkeys(macros)) # Eliminar duplicados manteniendo orden

df["artist_clean"] = df["artist"].apply(clean_artist)
df["genre_1_clean"] = df["genre_1"].apply(lambda x: get_macro_genres(x)[0] if get_macro_genres(x) else None)
df["genre_2_clean"] = df["genre_2"].apply(lambda x: get_macro_genres(x)[0] if get_macro_genres(x) else None)
df["genre_3_clean"] = df["genre_3"].apply(lambda x: get_macro_genres(x)[0] if get_macro_genres(x) else None)

def build_genre_list(row):
    genres = [row[c] for c in ["genre_1_clean", "genre_2_clean", "genre_3_clean"] if pd.notna(row[c])]
    return list(dict.fromkeys(genres))

df["genre_list"] = df.apply(build_genre_list, axis=1)
df["n_genres"] = df["genre_list"].apply(len)
df["word_count"] = df["lyrics"].apply(lambda x: len(re.findall(r"\b[\w']+\b", str(x))) if pd.notna(x) else 0)

# Filtrar canciones sin letra o sin género
mask = (df["lyrics"].notna()) & (df["word_count"] > 0) & (df["n_genres"] > 0)
df_clean = df[mask].copy()

# ============================================================
# 5. DEDUPLICACIÓN Y EXPLOSIÓN (LONG FORMAT)
# ============================================================
# Normalizar para detectar duplicados
df_clean["artist_norm"] = df_clean["artist"].str.lower().str.strip()
df_clean["song_norm"] = df_clean["song"].str.lower().str.strip()

# Eliminar duplicados quedándonos con la letra más larga
df_clean = (df_clean.sort_values("word_count", ascending=False)
            .drop_duplicates(subset=["artist_norm", "song_norm"], keep="first")
            .sort_index())

# Expandir: Una fila por cada género de la canción
df_genres = df_clean.explode("genre_list").rename(columns={"genre_list": "genre"})
df_genres = df_genres[df_genres["genre"].notna()]

# Guardar resultado
final_path = OUTPUT_DIR / "songs_clean_by_genre_exploded_transformer.csv"
df_genres.to_csv(final_path, index=False)

print(f"\nPROCESO FINALIZADO")
print(f"Filas resultantes (canción-género): {len(df_genres)}")
print(f"Archivo guardado en: {final_path}")

Shape inicial: (10081, 8)
Cargando modelo Transformer...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Detectados 1259 géneros únicos. Iniciando clasificación...


Clasificando géneros:   0%|          | 0/1259 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



PROCESO FINALIZADO
Filas resultantes (canción-género): 9189
Archivo guardado en: outputs\songs_clean_by_genre_exploded_transformer.csv


In [ ]:
df_letras_bien=pd.read_csv("outputs/songs_clean_by_genre_exploded_transformer.csv")


NameError: name 'head' is not defined

In [15]:
df_letras_bien.head()

,artist,song,lyrics,genres,genre_1,genre_2,genre_3,artist_clean,genre_1_clean,genre_2_clean,genre_3_clean,genre,n_genres,word_count,artist_norm,song_norm
0,/moonspell/,Heartshaped Abyss,Never resist\nThe firewalker\nHe has been sent...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,moonspell,Metal,Metal,Metal,Metal,1,167,/moonspell/,heartshaped abyss
1,/moonspell/,Let The Children Cum To Me...,"Hey you little Jesus' bride, why have you smil...","gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,moonspell,Metal,Metal,Metal,Metal,1,197,/moonspell/,let the children cum to me...
2,/moonspell/,Memento Mori,"I am the moment, the soul\nThe moment that end...","gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,moonspell,Metal,Metal,Metal,Metal,1,219,/moonspell/,memento mori
3,/moonspell/,Once It Was Ours!,Vanishing act inside the weak\nIn need of you ...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,moonspell,Metal,Metal,Metal,Metal,1,174,/moonspell/,once it was ours!
4,/moonspell/,Serpent Angel,Father Satan send the Serpent\nPoison me with ...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,moonspell,Metal,Metal,Metal,Metal,1,152,/moonspell/,serpent angel


In [22]:
# 1. Definir las columnas que queremos mantener
# Nota: He añadido 'song' (asumiendo que quieres el título original) 
# y he usado 'genre' que es el nombre que aparece en tu imagen.
columnas_finales = [
    'lyrics', 
    'artist_clean', 
    'genre', 
    'word_count', 
    'song_norm', 
    'song'
]

# 2. Filtrar el dataframe
df_final = df_genres[columnas_finales].copy()

# 3. (Opcional) Renombrar 'genre' a 'genre_clean' para que coincida con tu preferencia
df_final = df_final.rename(columns={'genre': 'genre_clean'})

# 4. Mostrar el resultado
print(f"Dataset final con {df_final.shape[1]} columnas.")
display(df_final.head())

# 5. Guardar si es necesario
# df_final.to_csv("outputs/dataset_final_reducido.csv", index=False)
len(df_final)

Dataset final con 6 columnas.


,lyrics,artist_clean,genre_clean,word_count,song_norm,song
0,Never resist\nThe firewalker\nHe has been sent...,moonspell,Metal,167,heartshaped abyss,Heartshaped Abyss
1,"Hey you little Jesus' bride, why have you smil...",moonspell,Metal,197,let the children cum to me...,Let The Children Cum To Me...
2,"I am the moment, the soul\nThe moment that end...",moonspell,Metal,219,memento mori,Memento Mori
3,Vanishing act inside the weak\nIn need of you ...,moonspell,Metal,174,once it was ours!,Once It Was Ours!
4,Father Satan send the Serpent\nPoison me with ...,moonspell,Metal,152,serpent angel,Serpent Angel


9189

In [24]:
# 1. Preparar el dataset de años (df_years)
df_years = pd.read_csv("NLP/datasets_scraping/dataset_final_con_años.csv")

# Función de normalización extrema para asegurar el cruce
def normalize_for_merge(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = html.unescape(text)
    # Eliminar barras, guiones y cualquier carácter no alfanumérico
    text = re.sub(r'[^a-z0-9]', '', text) 
    return text

# Crear columnas temporales de cruce en el dataset de años
df_years['artist_key'] = df_years['artist'].apply(normalize_for_merge)
df_years['song_key'] = df_years['song'].apply(normalize_for_merge)

# 2. Crear columnas temporales de cruce en tu dataset actual (df_final)
# Usamos artist_clean porque ya le quitaste las barras '/'
df_final['artist_key'] = df_final['artist_clean'].apply(normalize_for_merge)
df_final['song_key'] = df_final['song'].apply(normalize_for_merge)

# 3. Realizar la unión (Merge) usando las llaves normalizadas
df_integrado = pd.merge(
    df_final,
    df_years[['artist_key', 'song_key', 'year']],
    on=['artist_key', 'song_key'],
    how='left'
)

# 4. Gestión de nulos y 'Unknown'
df_integrado['year'] = pd.to_numeric(df_integrado['year'], errors='coerce')

# --- ANTES DE BORRAR: ¿Cuántos perdimos? ---
sin_año = df_integrado['year'].isna().sum()
total = len(df_integrado)
print(f"Canciones sin año encontrado: {sin_año} de {total} ({sin_año/total:.1%})")

# 5. Limpiar y eliminar auxiliares
df_integrado = df_integrado.dropna(subset=['year'])
df_integrado['year'] = df_integrado['year'].astype(int)
df_integrado = df_integrado.drop(columns=['artist_key', 'song_key'])

display(df_integrado.head())

Canciones sin año encontrado: 1382 de 9315 (14.8%)


,lyrics,artist_clean,genre_clean,word_count,song_norm,song,year
1,"Hey you little Jesus' bride, why have you smil...",moonspell,Metal,197,let the children cum to me...,Let The Children Cum To Me...,2008
2,"I am the moment, the soul\nThe moment that end...",moonspell,Metal,219,memento mori,Memento Mori,2020
3,Vanishing act inside the weak\nIn need of you ...,moonspell,Metal,174,once it was ours!,Once It Was Ours!,2020
4,Father Satan send the Serpent\nPoison me with ...,moonspell,Metal,152,serpent angel,Serpent Angel,2024
5,"And we travel into the darkening,\na couldn't ...",moonspell,Metal,188,the darkening,The Darkening,2003


In [25]:
len(df_integrado)

7933

In [26]:
from pathlib import Path

# 1. Asegurarnos de que la carpeta de destino existe
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# 2. Definir el nombre del archivo
nombre_archivo = "songs_with_years_processed.csv"
ruta_final = output_dir / nombre_archivo

# 3. Guardar el CSV
# index=False evita que se guarde una columna extra con los números de fila
df_integrado.to_csv(ruta_final, index=False, encoding='utf-8')

print(f"✅ ¡Hecho! Dataset guardado con éxito en: {ruta_final}")
print(f"Total de registros exportados: {len(df_integrado)}")

✅ ¡Hecho! Dataset guardado con éxito en: outputs\songs_with_years_processed.csv
Total de registros exportados: 7933
